Q1. Data Understanding

Identify all data quality issues present in the dataset that can cause problems during data loading.

1. Duplicate Records:
The record for Order_ID O101 appears twice (Rows 1 and 4).

2. Missing Values:
In Row 2 (Order_ID O102), the Sales_Amount is explicitly listed as "Null".

3. Inconsistent Data Types:
In Row 5 (Order_ID O104), the Sales_Amount is written as "Three Thousand" (string/text) instead of a numeric value like 3000.

4. Non-Standard Date Formats:
The Order_Date column lacks a uniform structure:

  Most rows use DD-MM-YYYY (e.g., 12-01-2024).

  Row 3 (O103) uses YYYY/MM/DD (2024/01/18).

Q2. Primary Key Validation

Assume order_id is the Primary Key.

a) Is the dataset violating the Primary Key rule?
b) Which record(s) cause this violation?

a) Is the dataset violating the Primary Key rule?

  Yes. A Primary Key constraint requires that every value in the column be unique and non-null. Since the value O101 appears more than once, the uniqueness requirement is violated.

b) Which record(s) cause this violation?

  The violation is caused by the duplicate entries for Order_ID: O101 in Row:1 and Row:4

Q3. Missing Value Analysis

Which column(s) contain missing values?

The Sales_Amount column contains an explicit missing value (represented as "Null").

a) List the affected records

The specific record affected by a missing value is:

Order_ID: O102 | Customer_ID: C002 | Sales_Amount: Null | Order_Date: 15-01-2024

b) Explain why loading these records without handling missing values is risky

Loading "Null" values into a production database or analytical tool poses several risks:

Calculation Failures: Mathematical functions like SUM(), AVERAGE(), or COUNT() behave inconsistently with nulls. In many systems, adding a number to a Null results in a Null, which would make your total sales reports incorrect.

Schema Violations: If the database table is configured with a "NOT NULL" constraint for the Sales_Amount column (which is common for financial data), the entire data loading process will crash and fail.

Inaccurate Business Insights: Missing data leads to an underestimation of total revenue. If 10% of your records have Null sales, your business KPIs (Key Performance Indicators) will be 10% lower than reality, leading to poor decision-making.

Application Crashes: Front-end applications or dashboards (like PowerBI or Tableau) may throw errors or display "NaN" (Not a Number) when trying to visualize data that contains unexpected empty fields.

Q4. Data Type Validation

Identify records where Sales_Amount violates expected data type rules.



a) Which record(s) will fail numeric validation?

The following records will fail when being processed as numeric data:

Order_ID O104: The value "Three Thousand" is a text string (string/varchar). A computer cannot perform mathematical operations on words without first converting them to digits.

Order_ID O102: The value "Null" often fails strict numeric validation if the system is not configured to handle empty values, though "Three Thousand" is the primary data type conflict.

b) What would happen if this dataset is loaded into a SQL table with Sales_Amount as DECIMAL?

If you attempt to load this data into a SQL table where the column is defined as DECIMAL, the following would occur:

Load Failure: The database engine will encounter the string "Three Thousand" and trigger a "Type Mismatch" or "Invalid Numeric Literal" error. In most standard SQL INSERT or BULK LOAD operations, this will cause the entire transaction to fail and roll back, meaning none of the rows (even the good ones) will be saved.

NULL Handling: For Order_ID O102, the result depends on the table schema:

If the column is DECIMAL NULL, it will load successfully as a database NULL.

If the column is DECIMAL NOT NULL, the load will fail at this row as well.

Truncation/Rejection: Some ETL (Extract, Transform, Load) tools might "quarantine" or reject these specific rows into an error log while loading the valid rows, but the raw SQL engine itself usually stops the process to ensure data integrity.

Q5. Date Format Consistency

The Order_Date column has multiple formats.



a) List all date formats present in the dataset

DD-MM-YYYY:12-01-2024

YYYY/MM/DD:2024/01/18

b) Why is this a problem during data loading?

Inconsistent date formats create a "silent killer" for data integrity. Here is why it's risky:

Parsing Failures: Most database loaders require a single, defined format string. When the loader hits Row 3 (2024/01/18) expecting DD-MM-YYYY, it will likely throw a "Value Error" and stop the entire import process.

Logical Corruption: If a date is written as 05-06-2024, a system expecting US format reads it as May 6th, while a system expecting European format reads it as June 5th. Without consistency, your time-series analysis will be factually wrong.

Sorting Issues: Dates stored as inconsistent strings cannot be sorted chronologically. 2024/01/18 would appear at the end of an alphabetical list, even though it happened before 25-01-2024.

Q6. Load Readiness Decision

Based on the dataset condition:

a) Should this dataset be loaded directly into the database? (Yes/No)

b) Justify your answer with at least three reasons

a) Should this dataset be loaded directly into the database? (Yes/No)

NO

b) Justify your answer with at least three reasons

Integrity Constraint Violations (Primary Key): The Order_ID O101 is duplicated. In a standard relational database, the Primary Key must be unique to identify a specific record. Loading without validation will trigger a Critical Error, causing the database to reject the entire batch to prevent data corruption.

Data Type Incompatibility (Schema Mismatch): The Sales_Amount column contains mixed data types, specifically the string "Three Thousand" in Row 5. A database column defined as DECIMAL or INT cannot translate English words into numbers automatically. This would result in a Parsing Error and a failed load.

Analytical Inaccuracy (Data Silos and Formatting): The inconsistent date formats (e.g., 12-01-2024 vs. 2024/01/18) and the presence of Null values would lead to "Silent Failures." Even if the data managed to load, your financial reports would be wrong—total revenue would be under-calculated due to the Null and the text-based sale amount, and chronological sorting would be broken.

Q7. Pre-Load Validation Checklist

List the exact pre-load validation checks you would perform on this dataset before loading.

1. Primary Key Uniqueness Check

2. Data Type & Schema Validation

3. Null Value Assessment

4. Date Format Standardization

5. Referential Integrity Check

Q8. Cleaning Strategy

Describe the step-by-step cleaning actions required to make this dataset load-ready.

The dataset contains four primary categories of errors that would cause a database upload to fail:

The Cleaning Strategy

Step1:

Duplicate Primary Keys: Order_ID O101 appears twice, violating uniqueness rules.

Cleaning Rule: Since the rows are identical, keep the first occurrence and delete the second. If the values differed, We would need to consult the source system to determine which record is the "source of truth."

Step2:

Data Type Mismatches: Sales_Amount contains a text string ("Three Thousand") where a number is expected.

Cleaning Rule: Manually or programmatically map text-based numbers to their numeric equivalents. Replace "Three Thousand" with the integer 3000.

Step3:

Missing Values: A Null entry in the sales column creates gaps in financial reporting.

Cleaning Rule: We have three options:

  1)Imputation: Replace the Null with a 0 or the column mean (risky for sales).

  2)Lookup: Check the original invoice to find the actual value.

  3)Removal: Delete the row if the missing data makes the record useless for analysis.

Step4:

Inconsistent Formatting: The Order_Date uses two different structures (DD-MM-YYYY and YYYY/MM/DD), making chronological sorting impossible.

Cleaning Rule:

Change 12-01-2024 to 2024-01-12.

Change 2024/01/18 to 2024-01-18.


Q9. Loading Strategy Selection

Assume this dataset represents daily sales data.

a) Should a Full Load or Incremental Load be used?

b) Justify your choice.

An Incremental Load should be used because this is daily sales data.Daily sales datasets grow over time. A Full Load requires deleting and reloading the entire historical table every day, which becomes slow and resource heavy as the business grows. An Incremental Load only appends the new data, saving processing power and time.

Q10. BI Impact Scenario

Assume this dataset was loaded without cleaning and connected to a BI dashboard.

a) What incorrect results might appear in Total Sales KPI?

b) Which records specifically would cause misleading insights?

c) Why would BI tools not detect these issues automatically?

a) What incorrect results might appear in Total Sales KPI?

Inflation via Duplication: The total sales figure would be inflated by 4,500 because the duplicate entry for O101 would be counted twice.

Underestimation via Type Mismatch: The dashboard would likely ignore the 3,000 from O104 because the BI tool treats "Three Thousand" as a text string.

Incomplete Revenue: The Null in O102 would result in a missing data point, further skewing the average sale per customer.

Result:KPI would show a "total" that is mathematically impossible to trust.

b) Which records specifically would cause misleading insights?

Order_ID: 0101(Duplicate):Artificially inflates Revenue and Customer Volume.

Order_ID: 0102(Null):Lowers the "Average Order Value" incorrectly.

Order_ID: 0103(Wrong Date):Might be excluded from "January 2024" reports if the tool cannot parse the YYYY/MM/DD format, leading to a dip in the monthly trend.

Order_ID: 0104(Text Value):Revenue is lost from the dashboard because strings cannot be summed.

c) Why would BI tools not detect these issues automatically?

The "GIGO" Principle: (Garbage In, Garbage Out). BI tools assume the data arriving from the database has already been cleaned. They prioritize displaying what they receive rather than questioning its accuracy.

Passive Data Handling: If a column contains both numbers and text (like "Three Thousand"), most BI tools will automatically convert the entire column to a String/Text type to prevent the application from crashing. Once it becomes a "string," you lose the ability to perform sums, averages, or any mathematical analysis.

Lack of Logic Awareness: A BI tool doesn't inherently know your business rules. It cannot distinguish between a legitimate repeat customer and a technical duplicate record unless a developer has manually built complex filters to check for it.

Parsing Limits: When faced with multiple date formats, the tool often fails to recognize the "odd one out" as a date at all, often categorizing it as "Null" or "Unknown," which removes those sales from your monthly trend lines.